In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

# find project root whether the notebook runs from root or from notebooks/
here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
assert RAW.exists(), f"data/raw not found from {here}"

YEAR = 2022
gdf = gpd.read_file(RAW / f"Latur_GT_ND_{YEAR}.shp")
print("fields:", len(gdf), "| CRS:", gdf.crs, "| columns:", len(gdf.columns))

fields: 55 | CRS: EPSG:4326 | columns: 47


In [3]:
pd.set_option("display.max_rows", None)

summary = []
for c in gdf.columns:
    if c == gdf.geometry.name:
        continue
    s = gdf[c]
    summary.append({
        "column": c,
        "dtype": str(s.dtype),
        "nulls": int(s.isna().sum()),
        "nunique": int(s.nunique(dropna=True)),
        "sample": str(list(s.dropna().unique()[:3]))[:60],
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

    column   dtype  nulls  nunique                                                       sample
       fid float64      0       55        [np.float64(6.0), np.float64(15.0), np.float64(17.0)]
       UID     str      0       55         ['2202-2397-1447', '1495-1526-580', '1502-1543-641']
      Farm     str      0       52                        ['Farm 1', 'Kaile rarm', 'Aute farm']
    Farmer     str      0       51   ['Jivanrao deshmukh', 'ramchandra  kaile', 'Baliram Aute']
     Phone     str      0       51                   ['9923762051', '9309863263', '9890580233']
 Survey_No     str     33       21                                          ['139', '3', '२२४']
     State     str      0        1                                              ['Maharashtra']
  District     str      0        1                                                    ['Latur']
    Taluka     str      0        4                                  ['Latur', 'Ausa', 'Chakur']
   Village     str      0       14      

In [4]:
# 2022 is two batches merged, so crop and area each got split into two columns.
# here we merge each pair back into one clean column.

# CROP: fill from whichever of the two crop columns has the value
crop = gdf["Crop_Name"].fillna(gdf["Crop Name"])

# AREA: use Area_acres where present; where it is empty, convert PolyArea (square meters) to acres
# 1 acre = 4046.86 square meters
area_acres = gdf["Area_acres"].fillna(gdf["PolyArea"] / 4046.86)

# check both are now fully filled (should be 55 / 55)
print("crop filled:", crop.notna().sum(), "/ 55")
print("area filled:", area_acres.notna().sum(), "/ 55")

crop filled: 55 / 55
area filled: 55 / 55


In [5]:
# Build a clean table. Keep everything with real signal, drop only junk.
# Feature selection comes LATER (during EDA), not here.

# soil description columns use "-" to mean "empty". Turn those into real blanks first.
soil_desc = gdf[["SColor", "SStructure", "STexture", "SDepth"]].replace("-", np.nan)

clean = gpd.GeoDataFrame({
    "uid": gdf["UID"],                 # field ID
    "year": YEAR,                      # which year (2022)

    # location
    "taluka": gdf["Taluka"],
    "village": gdf["Village"],

    # crop info
    "crop": crop,                      # merged crop column
    "variety": gdf["A_Variety"],
    "irrigation": gdf["Irrigation"],

    # target
    "crop_yield": pd.to_numeric(gdf["C_Yeild"], errors="coerce"),   # yield as number

    # dates (text -> real dates; "N/A" becomes blank automatically)
    "sowing_date": pd.to_datetime(gdf["ASowing"], dayfirst=True, errors="coerce"),
    "harvest_date": pd.to_datetime(gdf["HA_Date"], errors="coerce"),

    # size
    "area_acres": area_acres,          # merged area (acres)

    # soil chemistry
    "carbon": gdf["Carbon"],
    "ph": gdf["pH"],
    "ec": gdf["EC"],
    "nitrogen": gdf["Nitrogen"],
    "phosphorus": gdf["Phosphorus"],
    "potassium": gdf["Potassium"],

    # soil physical description
    "soil_color": soil_desc["SColor"],
    "soil_structure": soil_desc["SStructure"],
    "soil_texture": soil_desc["STexture"],
    "soil_depth": soil_desc["SDepth"],

    # map shape
    "geometry": gdf.geometry,
}, crs=gdf.crs)

print("new table size (rows, columns):", clean.shape)
print()
print("blanks in each column:")
print(clean.drop(columns="geometry").isna().sum())

new table size (rows, columns): (55, 22)

blanks in each column:
uid                0
year               0
taluka             0
village            0
crop               0
variety           17
irrigation         0
crop_yield         0
sowing_date       17
harvest_date       0
area_acres         0
carbon             0
ph                 0
ec                 0
nitrogen           0
phosphorus         0
potassium          0
soil_color        22
soil_structure    22
soil_texture      22
soil_depth        22
dtype: int64


C:\Users\Predator\AppData\Local\Temp\ipykernel_68980\2692354223.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  "harvest_date": pd.to_datetime(gdf["HA_Date"], errors="coerce"),


In [6]:
# Flag WRONG values only (not missing ones). We mark, never delete.
# A blank is a gap, handled later. Here we catch values that are impossible.

flags = []
for _, r in clean.iterrows():
    f = []

    # yield impossible (<=0) or too high for soybean (>20 quintal/acre)
    y = r["crop_yield"]
    if pd.notna(y) and (y <= 0 or y > 20):
        f.append("yield_suspect")

    # sowing date whose year is not this file's year (typo like 1960)
    sd = r["sowing_date"]
    if pd.notna(sd) and sd.year != YEAR:
        f.append("bad_sowing_year")

    # area zero or negative (impossible)
    a = r["area_acres"]
    if pd.notna(a) and a <= 0:
        f.append("area_suspect")

    flags.append(";".join(f))

clean["qc_flag"] = flags

# summary
print("flagged rows:", (clean["qc_flag"] != "").sum(), "of", len(clean))
print(clean["qc_flag"].value_counts())
print()

flagged = clean[clean["qc_flag"] != ""]
if len(flagged):
    print(flagged[["uid", "village", "crop_yield", "sowing_date", "area_acres", "qc_flag"]].to_string(index=False))
else:
    print("No wrong values found in 2022. Yields, dates and areas all look valid.")

flagged rows: 0 of 55
qc_flag
    55
Name: count, dtype: int64

No wrong values found in 2022. Yields, dates and areas all look valid.


In [7]:
# Save the cleaned 2022 table to data/interim. Raw stays untouched.
INTERIM.mkdir(parents=True, exist_ok=True)
out = INTERIM / f"clean_{YEAR}.gpkg"
clean.to_file(out, driver="GPKG")

print("saved:", out)
print("rows:", len(clean), "| columns:", clean.shape[1])
print("flagged (wrong values):", (clean["qc_flag"] != "").sum())
print("PII columns present:", [c for c in clean.columns if c in ("farmer", "phone", "agent")])  # should be []

saved: f:\THESIS\mtech_thesis_crop_yield_estimation\data\interim\clean_2022.gpkg
rows: 55 | columns: 23
flagged (wrong values): 0
PII columns present: []
